## MarketWatch Scraper

In [8]:
import requests
import bs4 as bs
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import duckdb as dd
from datetime import datetime
import time
import lxml


DB_PATH = "marketwatch_news.duckdb"
mw_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Origin': 'https://www.marketwatch.com',
        'Referer': 'https://www.marketwatch.com/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-User': '?1',
        'Sec-Fetch-Dest': 'document',
        'Sec-CH-UA': '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        'Sec-CH-UA-Mobile': '?0',
        'Sec-CH-UA-Platform': '"Windows"',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    }
delay_seconds = float(5)
session = requests.Session()
session.headers.update(mw_headers)
_session_primed = False

def init_db(db_path: str = DB_PATH):
    conn = dd.connect(db_path)
    conn.execute(
        '''
        DROP TABLE IF EXISTS marketwatch_articles;

        CREATE TABLE IF NOT EXISTS marketwatch_articles (
            query TEXT,
            title TEXT,
            url TEXT,
            subhead TEXT,
            snippet TEXT,
            author TEXT,
            published_at TIMESTAMP,
            last_updated_at TIMESTAMP,
            scraped_at TIMESTAMP
        );
        '''
    )
    conn.close()


def build_search_url(query: list) -> str:
    """
    Use MarketWatch search, sorted by recency.
    This targets both tickers and company names.
    """
    q = quote_plus(" OR ".join(query))
    # Basic search URL; you can refine with more params if needed
    ## return f"https://www.marketwatch.com/search?q={q}&page={page}&sort=recency"
    return f"https://www.marketwatch.com/search?q={q}&ts=2&m=0&sm=0&tab=Articles"
    ## LIST EXAMPLE URL: https://www.marketwatch.com/search?q=TSLA%20OR%20MSFT%20OR%20GOOG%20OR%20V&ts=2&m=0&sm=0&tab=Articles


def prime_session():
    global _session_primed
    if _session_primed:
        return
    try:
        resp = session.get("https://www.marketwatch.com/", timeout=15)
        resp.raise_for_status()
    except requests.RequestException:
        pass
    _session_primed = True


def fetch_search_page(query: list) -> BeautifulSoup:
    global session, _session_primed
    url = build_search_url(query)
    prime_session()

    # Use the global session so cookies and headers persist between homepage
    # requests and search requests. This reduces the chance of a 401 Forbidden.
    session.headers.update({
        "Referer": "https://www.marketwatch.com/",
        "Origin": "https://www.marketwatch.com",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-User": "?1",
        "Sec-Fetch-Dest": "document",
        "Sec-CH-UA": '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": '"Windows"',
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    })

    try:
        session.get("https://www.marketwatch.com/", timeout=15)
    except requests.RequestException:
        pass

    resp = session.get(url, timeout=15, allow_redirects=True)
    if resp.status_code == 401:
        # If the site still returns 401, reset the global session and retry
        # with a fresh browser-like session and homepage cookies.
        session = requests.Session()
        session.headers.update(mw_headers)
        _session_primed = False
        prime_session()
        try:
            session.get("https://www.marketwatch.com/", timeout=15)
        except requests.RequestException:
            pass
        resp = session.get(url, timeout=15, allow_redirects=True)

    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def fetch_article_page(url: str) -> BeautifulSoup:
    prime_session()
    if not url.startswith("http"):
        url = f"https://www.marketwatch.com{url}"
    session.headers.update({"Referer": "https://www.marketwatch.com/"})
    resp = session.get(url, timeout=15, allow_redirects=True)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def parse_search_results(soup: BeautifulSoup, delay_seconds: float = delay_seconds):
    """
    Parse MarketWatch search results and article pages.

    NOTE: MarketWatch may change its HTML structure.
    Inspect the page in your browser and adjust selectors if needed.
    """
    articles = []

    # Common pattern: result cards under something like:
    # <div class="searchresult"> or <div class="article__content"> etc.
    # We'll try a couple of patterns.
    result_containers = soup.select("div.searchresult, div.article__content")

    for container in result_containers:
        # Title and URL
        a = container.find("a", href=True)
        if not a:
            continue

        title = a.get_text(strip=True)
        url = str(a["href"])
        subhead = None
        snippet = None
        author = None
        published_at = None
        last_updated_at = None
        try:
            article_page = fetch_article_page(url)

            # Subhead
            subhead_el = article_page.select_one('h2[class*="article__subhead"]')
            subhead = subhead_el.get_text(strip=True) if subhead_el else None

            # Snippet
            snippet_el = article_page.select_one('section[class*="Container"]')
            snippet = snippet_el.get_text(strip=True) if snippet_el else None

            # Author
            author_el = article_page.select_one('a[data-testid*="author-link"]')
            author = author_el.get_text(strip=True) if author_el else None

            #Published Timestamp
            published_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="first"]>time')
            published_at = published_time_el['datetime'] if published_time_el else None

            #Last Updated Timestamp
            last_updated_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="last"]>time')
            last_updated_at = last_updated_time_el['datetime'] if last_updated_time_el else None
        
        except:
            # If fetching/parsing article page fails, we can still save basic info
            pass

        articles.append(
            {
                "title": title,
                "url": url,
                "subhead": subhead,
                "snippet": snippet,
                "author": author,
                "published_at": published_at,
                "last_updated_at": last_updated_at
            }
        )

        time.sleep(delay_seconds)  # be polite

    return articles


def save_articles_to_duckdb(
    articles,
    db_path: str = DB_PATH,
):
    conn = dd.connect(db_path)
    scraped_at = datetime.utcnow()
    
    for art in articles:
        conn.execute(
            """
            INSERT INTO marketwatch_articles
                (title, url, subhead, snippet, author, published_at, last_updated_at, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            [
                art.get("title"),
                art.get("url"),
                art.get("subhead"),
                art.get("snippet"),
                art.get("author"),
                art.get("published_at"),
                art.get("last_updated_at"),
                scraped_at,
            ],
        )

    conn.close()


def scrape_marketwatch_for_query(
    query: list,
    pages: int = 1,
    delay_seconds: float = delay_seconds,
):
    """
    Scrape MarketWatch search results for a ticker or company name
    and store them into DuckDB.
    """
    all_articles = []

    for page in range(1, pages + 1):
        ## print(f"Fetching page {page} for query '{query}'...")
        soup = fetch_search_page(query)
        articles = parse_search_results(soup)
        ## print(f"  Found {len(articles)} articles on page {page}.")
        if articles not in all_articles:
            all_articles.extend(articles)
        time.sleep(delay_seconds)  # be polite
    
    save_articles_to_duckdb(all_articles)  
    print(f"Saved {len(all_articles)} articles for query '{query}'.")


## SCRAPE S&P 500 TICKERS

In [9]:
s_p_500_ticker_url = 'http://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
wiki_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Origin': 'https://en.wikipedia.org',
        'Referer': 'https://en.wikipedia.org/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-User': '?1',
        'Sec-Fetch-Dest': 'document',
        'Sec-CH-UA': '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        'Sec-CH-UA-Mobile': '?0',
        'Sec-CH-UA-Platform': '"Windows"',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    }
resp = requests.get(s_p_500_ticker_url, headers=wiki_headers)
soup = bs.BeautifulSoup(resp.text, 'html.parser')
table = soup.select_one('table[id*="constituents"]')
tickers = [row.findAll('td')[0].text.strip() for row in table.findAll('tr')[1:]]

C:\Users\Dsavs\AppData\Local\Temp\ipykernel_11844\2681278560.py:24: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tickers = [row.findAll('td')[0].text.strip() for row in table.findAll('tr')[1:]]


In [10]:
tickers[0:5]

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']

## Run Main Program

In [14]:
if __name__ == "__main__":
    # 1. Initialize DB (run once)
    init_db()

    # 2. Example: scrape daily news for a ticker
    #    You can schedule this script via cron/Task Scheduler to run daily.
    ## tickers = ['TSLA', 'META', 'BRK.A', 'V', 'UNH', 'HD', 'PG', 'MA', 'DIS', 'CMCSA',
    ##            'PFE' , 'VZ', 'KO', 'MRK', 'IBM', 'INTC', 'CSCO', 'ADBE', 'NFLX', 'PYPL',
    ##            'PEP', 'NVDA', 'ORCL', 'ABNB', 'CRM', 'T', 'XOM', 'CVX', 'WMT', 'BA', 'NKE']
              
    ticker_batch_size = 50
    ## company_name = "Microsoft"   # you can also use company names

    # Scrape by ticker
    for i in range(0, len(tickers), ticker_batch_size):
        batch = tickers[i:i + ticker_batch_size]
        scrape_marketwatch_for_query(batch)

    # Scrape by company name (optional)
    ## scrape_marketwatch_for_query(company_name, pages=1)

HTTPError: 401 Client Error: HTTP Forbidden for url: https://www.marketwatch.com/search?q=MMM+OR+AOS+OR+ABT+OR+ABBV+OR+ACN+OR+ADBE+OR+AMD+OR+AES+OR+AFL+OR+A+OR+APD+OR+ABNB+OR+AKAM+OR+ALB+OR+ARE+OR+ALGN+OR+ALLE+OR+LNT+OR+ALL+OR+GOOGL+OR+GOOG+OR+MO+OR+AMZN+OR+AMCR+OR+AEE+OR+AEP+OR+AXP+OR+AIG+OR+AMT+OR+AWK+OR+AMP+OR+AME+OR+AMGN+OR+APH+OR+ADI+OR+AON+OR+APA+OR+APO+OR+AAPL+OR+AMAT+OR+APP+OR+APTV+OR+ACGL+OR+ADM+OR+ARES+OR+ANET+OR+AJG+OR+AIZ+OR+T+OR+ATO&ts=2&m=0&sm=0&tab=Articles

## Clean DuckDB persistent database file

In [12]:
## import duckdb as dd
con = dd.connect(DB_PATH)  # Open the persistent database
con.execute('''
            DROP TABLE IF EXISTS marketwatch_articles_deduped;
            
            CREATE TABLE IF NOT EXISTS marketwatch_articles_deduped AS
            SELECT DISTINCT query, title, subhead, snippet,
            coalesce(last_updated_at, published_at) AS article_date
            FROM marketwatch_articles
            WHERE
            coalesce(last_updated_at, published_at) is not null
            --AND (url LIKE '%barrons.com%' or url LIKE '%marketwatch.com%')
            ;

            DROP TABLE IF EXISTS marketwatch_articles;
            '''
)
con.close()

In [13]:
con = dd.connect(DB_PATH)  # Open the persistent database
result = con.sql('''
                 SELECT *
                 FROM marketwatch_articles_deduped
                 ;
                 '''
                 ).df()
con.close()
result

,query,title,subhead,snippet,article_date
